# Demo RAG đơn giản với Nội dung text

Notebook này minh hoạ **RAG (Retrieval-Augmented Generation)** từ đầu đến cuối, không dùng framework RAG có sẵn (LangChain/LlamaIndex), để bạn thấy rõ từng bước bên trong.

**Luồng xử lý (pipeline):**
1. Cài đặt thư viện (Langchain)
2. Chia nhỏ text thành các đoạn (chunking)
3. Biến mỗi đoạn thành vector số (embedding)
4. Lưu các vector vào một chỉ mục tìm kiếm (chroma index)
5. Khi có câu hỏi: embed câu hỏi -> tìm các đoạn liên quan nhất (retrieval)
6. Ghép các đoạn tìm được + câu hỏi vào prompt -> đưa cho LLM sinh câu trả lời (generation)

Model dùng trong demo này chạy **local, miễn phí**, không cần API key, phù hợp Colab free tier.

## **Bước 1: Cài đặt thư viện và Nhập API Key**

In [6]:
!pip install -U \
    langchain \
    langchain-core \
    langchain-text-splitters \
    langchain-google-genai \
    langchain-chroma \
    chromadb

In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Nhập Gemini API Key: ")

Nhập Gemini API Key: ··········


In [7]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_chroma import Chroma

print("Import OK!")

Import OK!


## **Bước 2: Chunking - chia text thành các đoạn nhỏ**

Vì sao phải chia nhỏ? Vì:
- LLM có giới hạn độ dài context, không thể nhét cả cuốn sách vào prompt
- Chia nhỏ giúp việc tìm kiếm (retrieval) chính xác hơn - chỉ lấy đúng phần liên quan

Ta dùng cách đơn giản: cắt theo số ký tự, có phần chồng lấn (overlap) giữa các đoạn để không mất ngữ cảnh ở ranh giới.

In [24]:
# Tạo dữ liệu mẫu là các đoạn text rời rạc
documents = [
    """
    RAG (Retrieval-Augmented Generation) là kỹ thuật kết hợp giữa
    retrieval và generative AI. Hệ thống sẽ tìm kiếm thông tin liên quan
    từ một knowledge base trước khi đưa thông tin đó cho LLM tạo câu trả lời.
    """,

    """
    Một pipeline RAG cơ bản gồm các bước:
    load dữ liệu, chunking, embedding, lưu vector vào vector database,
    retrieval các tài liệu liên quan và cuối cùng đưa context cho LLM.
    """,

    """
    Embedding chuyển văn bản thành một vector số.
    Các văn bản có ý nghĩa tương tự nhau thường có vector nằm gần nhau.
    Embedding được sử dụng để semantic search và information retrieval.
    """,

    """
    Chroma là một vector database có thể lưu embedding và thực hiện
    similarity search. Trong hệ thống RAG, Chroma được sử dụng để
    tìm những document gần nhất với câu hỏi của người dùng.
    """,

    """
    LLM nhận câu hỏi cùng với context được retriever tìm ra.
    LLM sử dụng context này để tạo câu trả lời.
    Nếu context không chứa thông tin cần thiết thì hệ thống nên nói
    rằng không tìm thấy thông tin trong tài liệu.
    """
]

docs = [
    Document(page_content=text)
    for text in documents
]

print("Số documents:", len(docs))

Số documents: 5


In [26]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print("Số chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)

Số chunks: 5

--- Chunk 0 ---
RAG (Retrieval-Augmented Generation) là kỹ thuật kết hợp giữa
    retrieval và generative AI. Hệ thống sẽ tìm kiếm thông tin liên quan
    từ một knowledge base trước khi đưa thông tin đó cho LLM tạo câu trả lời.

--- Chunk 1 ---
Một pipeline RAG cơ bản gồm các bước:
    load dữ liệu, chunking, embedding, lưu vector vào vector database,
    retrieval các tài liệu liên quan và cuối cùng đưa context cho LLM.

--- Chunk 2 ---
Embedding chuyển văn bản thành một vector số.
    Các văn bản có ý nghĩa tương tự nhau thường có vector nằm gần nhau.
    Embedding được sử dụng để semantic search và information retrieval.

--- Chunk 3 ---
Chroma là một vector database có thể lưu embedding và thực hiện
    similarity search. Trong hệ thống RAG, Chroma được sử dụng để
    tìm những document gần nhất với câu hỏi của người dùng.

--- Chunk 4 ---
LLM nhận câu hỏi cùng với context được retriever tìm ra.
    LLM sử dụng context này để tạo câu trả lời.
    Nếu context không ch

## **Bước 3: Tạo embedding cho từng đoạn**

Embedding là cách biến 1 đoạn text thành 1 vector số (ví dụ 384 chiều), sao cho các đoạn có nghĩa gần nhau thì vector cũng gần nhau. Ta dùng model `all-MiniLM-L6-v2` - nhỏ, nhanh, miễn phí, chạy tốt trên CPU.

In [10]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

print("Embedding model:", embeddings.model)

Embedding model: gemini-embedding-001


## **Bước 4: Lưu embeddings vào VectorDataBase**

Ta lưu tất cả vector đoạn văn vào đây để sau này tìm kiếm.

In [11]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="gemini_rag_demo"
)

print("Vector database đã được tạo!")

Vector database đã được tạo!


## **Bước 5: Hàm Retrieval - tìm đoạn liên quan nhất tới câu hỏi**

Với 1 câu hỏi bất kỳ: embed câu hỏi đó, rồi tìm k đoạn có vector gần nhất trong index.

In [12]:
question = "RAG là gì?"

results = vectorstore.similarity_search(
    question,
    k=3
)

for i, doc in enumerate(results):
    print(f"\n===== RESULT {i+1} =====")
    print(doc.page_content)


===== RESULT 1 =====
RAG (Retrieval-Augmented Generation) là kỹ thuật kết hợp giữa
    retrieval và generative AI. Hệ thống sẽ tìm kiếm thông tin liên quan
    từ một knowledge base trước khi đưa thông tin đó cho LLM tạo câu trả lời.

===== RESULT 2 =====
Một pipeline RAG cơ bản gồm các bước:
    load dữ liệu, chunking, embedding, lưu vector vào vector database,
    retrieval các tài liệu liên quan và cuối cùng đưa context cho LLM.

===== RESULT 3 =====
Chroma là một vector database có thể lưu embedding và thực hiện
    similarity search. Trong hệ thống RAG, Chroma được sử dụng để
    tìm những document gần nhất với câu hỏi của người dùng.


## **Bước 6: Load LLM để sinh câu trả lời**

Ta dùng `gemini-3.6-flash` - model nhỏ, chạy được trên CPU/GPU free của Colab, không cần API key. (Nếu bạn có API key OpenAI/Claude, có thể thay bước này bằng gọi API để chất lượng trả lời tốt hơn nhiều - xem ghi chú cuối notebook.)

In [19]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

## **Bước 7: Ghép prompt và sinh câu trả lời (Generation)**

Đây là bước "Augmented Generation": ta đưa các đoạn văn tìm được làm **context**, kèm câu hỏi, vào 1 prompt duy nhất, rồi để LLM trả lời dựa trên context đó thay vì chỉ dựa vào kiến thức có sẵn của nó.

In [27]:
prompt = ChatPromptTemplate.from_template("""
Bạn là một trợ lý AI sử dụng RAG.

Hãy trả lời câu hỏi CHỈ dựa trên context bên dưới.

Nếu context không chứa đủ thông tin để trả lời,
hãy nói:

"Tôi không tìm thấy thông tin này trong tài liệu."

Không được tự bịa thông tin.

=== CONTEXT ===
{context}

=== QUESTION ===
{question}

=== ANSWER ===
""")

## **Bước 8: Thử hỏi thoải mái**

Sửa câu hỏi bên dưới theo nội dung file PDF bạn đã upload.

In [18]:
def rag(question, k=3):

    # 1. Retrieve
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=k
    )

    # 2. Tạo context
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # 3. Tạo prompt
    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    # 4. Gọi Gemini
    response = llm.invoke(messages)

    return {
        "answer": response.content,
        "documents": retrieved_docs
    }

In [20]:
result = rag("RAG là gì?")

print("===== ANSWER =====")
print(result["answer"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


===== ANSWER =====
[{'type': 'text', 'text': 'RAG (Retrieval-Augmented Generation) là kỹ thuật kết hợp giữa retrieval và generative AI. Hệ thống sẽ tìm kiếm thông tin liên quan từ một knowledge base trước khi đưa thông tin đó cho LLM tạo câu trả lời.', 'extras': {'signature': 'EqgPCqUPARFNMg8slyNzJWZEsBJ5OswgLDMBeultySE63gdIqxCHxIlyHWGGtwbB6YMf5ic8i+P9ua1qqc2Jk3N5j5v18nS++A1NlOybbzcHf/lvR7s631G91KWmVqEHZAnuVqilCucNtwYjI1OXDkPRAbHnPRzVndz+p79g6bYTTp6pEsPVveJ7RHqHGHjsXHuaUMT3W7OzFTyUiOjZspYBecjZsKQuqRfz5Y3sHe6iEeUh68rGN22DM9Unvc2znacoZcnV9Z2AK4A1RJicyub1+u/YVpgYblT0c0iG3ZlQXM1MYnz4fCMcIUPup9j8DEelC/xcJLeCpCVSdWJDDpSAsa2oNFxPhdzMcTw9ZXNHJ/Gp2lumkcLMuvJi8bxPblpNlBS6lV/uPNZJj71bWbj4wD7IbtW/UX8LyYcghdHstMDRKtF3EUH/ZcIfIVIp+AsnhjvkFmaxi8YHRYZZpO/ED2zBpqByPtI0wZzucrKs5HugiItn8f76w29ui0rmdAAwcFK05c2R7GDIIgLHwRqrzxwE4cnm2v+ZdXJSwa9wT65+RmJR4K4U6F9SWh03SlvALsKP3fjj/FAZB8SfUCHKD8m0Yr7UpeUDOh7ESb0vQvdxNVrCC7o5S7F8apr/tBKFTxrycSTi6sj3flBIp128AfJVhDKmn6B2Bzwu0GsyiXt1gIGDIqjtpy3vZXhzFWIKjB+nnAYK/QsSfT

In [21]:
questions = [
    "Embedding dùng để làm gì?",
    "Chroma có vai trò gì trong RAG?",
    "Pipeline RAG gồm những bước nào?"
]

for question in questions:

    result = rag(question)

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("-" * 70)
    print("ANSWER:", result["answer"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



QUESTION: Embedding dùng để làm gì?
----------------------------------------------------------------------
ANSWER: [{'type': 'text', 'text': 'Dựa trên tài liệu được cung cấp, Embedding được sử dụng để:\n- Chuyển văn bản thành một vector số.\n- Thực hiện tìm kiếm ngữ nghĩa (semantic search) và truy xuất thông tin (information retrieval).', 'extras': {'signature': 'EoINCv8MARFNMg+pAaMCmFhL+elt7ihlWTplKkC1Tovujt+/HJwKRh8E0Az+FLdPl3LEHY0r3YFO2cknh/uofHZeVovCKA6E8eUDff9OSbifwJuF0zwDs2VUKHMZauVjJPV8zWzOQKx02KebK1GQpeb3B8fx9aq/XxBA4Dwz1W0PbwF5jZuAf18hsXuexqVcHhicPVqjXBG9JxKnJjLnFiMMWhylLS5h6ZitbgN0Obzc+Z6U0/dfHF0goI4s5LArAsf+OY74+4uHJMUHcWZQcyMHyL8QrtEFVgNbkii7D0l8CGDotALi157OtXtt6+3rI+X5nerJymZBcOcHKTKPuTXHUU/RWKuOkSbGYltyJ/lSEiNaS4VERN/SMukzTOl54lXGMzJr97mJoLIiMlTqjNIlCk9kY6KU3XI/5i7GQ6f4K164iNIbLip+DXYOJ6d6emNI6s2uL3d2EqKgEB1JxXMrldU78dtK9Nsq7N3brOMbpAkrbh0n95S8AqSrKDTnYDSvfz8lCFQGtBhfXwNYODtDDyYzBtSXMgw/zsXA9opySTq7MLU7l2c1OnNBL6nSUqEfheStwvYnElk4+vxmbfd3QoK1fcAOE36TvdrXwB9OwABNVoZ1G9Mfb

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



QUESTION: Chroma có vai trò gì trong RAG?
----------------------------------------------------------------------
ANSWER: [{'type': 'text', 'text': 'Trong hệ thống RAG, Chroma là một vector database có vai trò lưu embedding, thực hiện similarity search và được sử dụng để tìm những document gần nhất với câu hỏi của người dùng.', 'extras': {'signature': 'EuEOCt4OARFNMg/fI7ERVj+Fe8lOt8PyIL9mARzWKUPIQHXFd4UX+n7sxclN0CgDUI9BQSrICzJ9Or2H3nymcZQbLRh7cMMtljhAxaktzQ76GwAQ5swHujyr72OTZe3wvvaCbOS7SUcWTkYu3fi6v5R5e2jQ4tFL95kzMMepe0k1EIlZWHk17/n++Y0E9RFmNOl7i4wOuarjEtFJ1c5JY0uiGPENxjMnAGWsVQsoQLxYe+U/0f+NWXMq4MgrKqTam5Itcr4s3sWBWqmoDzS5EegPrkYKCdbDTpv05fVck44wMCWN/UirlqzdpyfMnisfTj62v0nqm3AWPxTfJlVJ0SinYzcCXiQThTZudKgF+XUPOaCdGVltQA5ucvyn8z+VX8vB5YFE3yqiUYXhKxxglpqx0qCBWYGdZAKfaCsTxAeo/k8LttaTKaiLlZ8vVDq9LvVX1052dWk3rMlpl8y9Bc/pylJ9HfhOI235AjSzt59pbvliYWgfE3GeLkR3D1cUBC0TfhTi8t8DkmXquzMq/mTVVKBjLwZgILYrbgkClAtPctavNk8eQg8rfcSx6o1QtiWgt3Fzetf4Jb9UDzBq1tL1tiXMUyw41zWw/s9wwwAOpGFsxEJOmYk1hQP+oZWTRvVdt

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



QUESTION: Pipeline RAG gồm những bước nào?
----------------------------------------------------------------------
ANSWER: [{'type': 'text', 'text': 'Một pipeline RAG cơ bản gồm các bước:\n1. Load dữ liệu\n2. Chunking\n3. Embedding\n4. Lưu vector vào vector database\n5. Retrieval các tài liệu liên quan\n6. Đưa context cho LLM', 'extras': {'signature': 'EoQOCoEOARFNMg8EwcS/WVPeoH9m/YyXaHZ+AEYZCd6Br6s2I6pmZZvFrau1XWbcho6xDbFdJ1WL44r1+MkJqeMucoxRFb5LT6FmXt+k+aN7o4OePCvMIaWVSaCt3feufH8Ekwu9pJK5AwAYjzhEuImpoltgOAOU/PcF+6AsdDeq0kr6v97s2mafRDPeK7CC5KXCgnJRn1SJFJQ8GLyJXBUvjaTLFPLsVmXXvBBGQ2MuNvgORaSXu6Fm43JPREnIBa671R1Umxv/itCYsMsMpDdcc7O1onbyaZrsyHEz/ukSxi0tMStN+6Q2Wk7in4zafOtZxJZZmvJ6kQd3Z2usj4lWCfe8y/jRiiQCfB7S4hFM0SjvLbRzokK/ndHB0hirCQ/aAb7Y9iNes9WxDSASjwg15lNmON4fsYeuQKt+25+PyW3ataspQuD6GmsVQ9Mhvhpcx9LP+qoOpPIBC21Vfj9RX8Ba7Bsjw1SgIfnzQSKCR4Kggd7rPz4lvSslA8eLbJd2yXMF4iikrCHXEPoAWTVTTh59z275P7WYInqr5Ov6vq8kNOaVKIYyCINaBntkHENa8eAp3aNCSwUQJnCMmXZX+tgnrVtxCYS5MjLrTAuuU5kN26AYEfXT3LCpuOwIZhqkp

In [22]:
question = "Chroma dùng để làm gì?"

result = rag(question, k=2)

print("===== QUESTION =====")
print(question)

print("\n===== ANSWER =====")
print(result["answer"])

print("\n===== RETRIEVED DOCUMENTS =====")

for i, doc in enumerate(result["documents"]):

    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


===== QUESTION =====
Chroma dùng để làm gì?

===== ANSWER =====
[{'type': 'text', 'text': 'Chroma là một vector database được dùng để lưu embedding và thực hiện similarity search. Trong hệ thống RAG, Chroma được sử dụng để tìm những document gần nhất với câu hỏi của người dùng.', 'extras': {'signature': 'EuoPCucPARFNMg+4Mnhy2qMfvd2lDZRWRcPOe1LysdE72q7Jlv+8F8ky0svpvaK8owFo5Pz3P0iDANZhOMUoYbrv60ytujMTGbDDJaZmDzrfbujZ72TuQnBJxc318yFbUIqeeWkeWovRb68e7gwog8Sj1Gah4Xp36pOLfJyvlqrrGz4vFLo8DOcFuzv1F3To4OUOAnKUlcrTOceVqLUebJuvytxQ0PaN+pqyjLQJrcuY2s73rDyg6IYfxOE4IDxDHtfgGdh9V7r60vt62mRC5SoNMoplSwkbDsmdgghATw4awNuu+teM4rxV6+vgrL/LpV+d5siW65DgWxefll4VVJDaAmYiGE1bdr6lG3lqPfs7GDjTc9Rk11FLTUzfYZEGYIeZy4UKyXWjgpOzwQmRuHYJvaLG7sCaCwDCdMJFTlJ5m36hbHjoGKa5llE0nvZeRa190TDgUNNHLjeknzHwAPqvP7D4VcX2v4Z3aza7B+MeeZEQwRHrrJaMnJrVJACG11SBIrf4yQZtMIL01L9TqCWjtPor5qmeVJqizSFhpY70Pjz7iSnKkpKbUAq57mGHHh3QxUZ4C5utDftWuIMcZ0cDxfgOn72kD26AMghICBo1bkPBK4h9ZGieoNksMu8zzkSrtoNS6sYOBvwloxaWfeaekgf0QFQo82tFyXhaHmKQ0DMd2WvYew

## **Bước 9: Kiểm tra Halluciation**


In [23]:
question = "Albert Einstein sinh năm bao nhiêu?"

result = rag(question)

print(result["answer"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Tôi không tìm thấy thông tin này trong tài liệu.', 'extras': {'signature': 'EqAKCp0KARFNMg/IEj0QhbeyGgz7E/E0EdGn7/BqvcxmuhlqbFfOgrAnC7MqO+WLIobzbLXJlcWxXltyS1uGTw6WYUz+UNRqsSxGDzv804qdAyWqe/iMjt9LgWtMAmaFMvPYGW9TbCxv5WuG/DiZXSY9HypDL18aBdJU7tyMnVGkGxcu/+dY6pXoV5d6mlwph0CcmLl7zau/oGLpYhIn7tTJjWrRpGbe0IswLisG4Wrus6KT0sIKPk0G22p+9R87Z+yMj0tRB952M8lqThmlv2NbGvCH9cNI57vEGUjGad3ccXGZkvUzCzl5U+1t9mV6+qp2YlSxqJTB/sIYbIoYmrr1ZaVJcvPkcw/ZMCJZZbeCxcgT/+anGRm3IZqiooALlk1zecOFFoS/piRfuG14V95HA6B5YQJ+3uTisvJcV/WQjzklBiWz3n9zha4R7yhXZOaXH/E+rYuH/hITjYl86zxPOZ4ACdL+6xRPV4A0+6L3KQFLFSEOsqvmy4cBXEvHAbyIOl5JxO3NCJ30zKylXPUoejqPAU+wR9d4MstRLW7uvZ6+c/HYlRdssHuDgHzirJgLV2uYrk+rc4r+1mJa4QmO06q7Fgc83c41JwlR6u7C9HHGJemEv3o3J3IfnjsIK1NTSExPaIScojlxcwIVwkAz2GDYEA7ykDzYN7HsmQYbarmwkEOH9zC1hQvAdd79JdFTCuqtKZcXpmvf0ANP+xZyGpBrrivz6dxLCaOhYyqf5Sp6OmELZy1IRbYJw/weYO33QlFgs3hQp7xya1N4m8Ak21hQyaUf+VInjla+b12dmMuEs1e+8UkXj845wFsI1IfaKNd9I+0ug7aDHa154JRgpfIVJVrvJiNkNEEbvcZXdrcBs0CyEMyYwdtFgSOjO